# Prompt Chaining and Sequencing Tutorial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/prompt-chaining-sequencing.ipynb)

## Overview

This tutorial explores the concepts of prompt chaining and sequencing in the context of working with large language models. We will use an open-source model (Qwen3-8B) running locally via HuggingFace Transformers in Google Colab to demonstrate how to connect multiple prompts and build logical flows for more complex AI-driven tasks.

## Motivation

As AI applications become more sophisticated, there is often a need to break down complex tasks into smaller, manageable steps. Prompt chaining and sequencing allow us to guide language models through a series of interrelated prompts, enabling more structured and controlled outputs. This approach is particularly useful for tasks that require multiple stages of processing or decision-making.

## Key Components

1. **Basic Prompt Chaining**: Connecting the output of one prompt to the input of another.
2. **Sequential Prompting**: Creating a logical flow of prompts to guide the AI through a multi-step process.
3. **Dynamic Prompt Generation**: Using the output of one prompt to dynamically generate the next prompt.
4. **Error Handling and Validation**: Implementing checks and balances within the prompt chain.

## Method Details

We will start by setting up our Google Colab environment with HuggingFace Transformers and loading a quantized open-source model. Then, we will explore basic prompt chaining by connecting two simple prompts. We will move on to more complex sequential prompting, where we guide the AI through a multi-step analysis process. Next, we will demonstrate how to dynamically generate prompts based on previous outputs. Finally, we will implement error handling and validation techniques to make our prompt chains more robust.

Throughout the tutorial, we use practical examples to illustrate these concepts, such as a multi-step text analysis task and a dynamic question-answering system.

## Conclusion

By the end of this tutorial, you will have a solid understanding of how to implement prompt chaining and sequencing using open-source models and native Python. These techniques will enable you to tackle more complex tasks, improve the coherence and relevance of AI-generated content, and create more interactive and dynamic AI-driven experiences.

## Chaining as a Computational Graph

A prompt chain is fundamentally a **directed acyclic graph (DAG)**. Each node is a generation step — a single call to the language model with a specific instruction. Each directed edge represents a data dependency: the output of one node becomes part of the input to the next.

This perspective is more than academic. When you model a chain as a DAG, you immediately gain insight into:

- **Parallelizable steps**: Nodes with no mutual dependency can execute concurrently. In the text-analysis example later in this notebook, "identify theme" and "identify tone" share no dependency on each other — they only depend on the original text. A DAG view makes this obvious.
- **Critical path**: The longest sequence of dependent steps determines minimum latency. Optimizing off-critical-path steps yields no latency benefit.
- **Bottlenecks**: If one node's output feeds many downstream nodes, its failure or slowness cascades. Identifying these hub nodes tells you where to invest in robustness (retries, caching, fallbacks).

Below, we build a simple DAG representation and print a text-based visualization to make these relationships explicit.

In [ ]:
# Represent a prompt chain as a DAG and visualize it as text

class ChainDAG:
    """A lightweight directed acyclic graph for prompt chains."""

    def __init__(self):
        self.nodes = {}       # name -> description
        self.edges = []       # (from, to)

    def add_node(self, name, description=""):
        self.nodes[name] = description

    def add_edge(self, src, dst):
        self.edges.append((src, dst))

    def predecessors(self, node):
        return [s for s, d in self.edges if d == node]

    def successors(self, node):
        return [d for s, d in self.edges if s == node]

    def roots(self):
        targets = {d for _, d in self.edges}
        return [n for n in self.nodes if n not in targets]

    def topo_sort(self):
        in_degree = {n: 0 for n in self.nodes}
        for s, d in self.edges:
            in_degree[d] += 1
        queue = [n for n in self.nodes if in_degree[n] == 0]
        order = []
        while queue:
            queue.sort()  # deterministic
            node = queue.pop(0)
            order.append(node)
            for s in self.successors(node):
                in_degree[s] -= 1
                if in_degree[s] == 0:
                    queue.append(s)
        return order

    def visualize(self):
        """Print a text-based DAG showing dependencies and parallelism."""
        in_deg = {n: 0 for n in self.nodes}
        for s, d in self.edges:
            in_deg[d] += 1

        # Assign levels (longest-path from root)
        levels = {n: 0 for n in self.nodes}
        for node in self.topo_sort():
            for succ in self.successors(node):
                levels[succ] = max(levels[succ], levels[node] + 1)

        # Group by level
        by_level = {}
        for n, lv in levels.items():
            by_level.setdefault(lv, []).append(n)

        print("=" * 60)
        print("PROMPT CHAIN DAG")
        print("=" * 60)
        for lv in sorted(by_level):
            nodes_at_level = sorted(by_level[lv])
            parallel_tag = " (parallelizable)" if len(nodes_at_level) > 1 else ""
            print(f"\nLevel {lv}{parallel_tag}:")
            for n in nodes_at_level:
                deps = self.predecessors(n)
                dep_str = f" <- depends on: {deps}" if deps else " (input/root)"
                print(f"  [{n}] {self.nodes[n]}{dep_str}")
            if lv < max(by_level):
                print("       |")
                print("       v")
        print("\n" + "=" * 60)

# Build the DAG for the text-analysis chain from later in this notebook
dag = ChainDAG()
dag.add_node("input",     "Receive raw text")
dag.add_node("theme",     "Identify main theme")
dag.add_node("tone",      "Describe overall tone")
dag.add_node("takeaways", "Extract key takeaways")

dag.add_edge("input", "theme")
dag.add_edge("input", "tone")
dag.add_edge("theme", "takeaways")
dag.add_edge("tone",  "takeaways")

dag.visualize()
print("\nTopological order:", dag.topo_sort())
print("\nNotice: 'theme' and 'tone' are at the same level — they could")
print("run in parallel since they only depend on 'input', not each other.")

### The Latency–Quality Tradeoff

Chains trade **latency** for **quality**. Every additional step adds wall-clock time (at minimum one full model forward pass), but each individual step is simpler and more focused than a monolithic prompt that tries to do everything at once.

Why does decomposition help quality?

1. **Reduced cognitive load per step.** A model generating 50 tokens for a narrow sub-task (e.g., "identify the tone") allocates all its attention to that sub-task, rather than juggling multiple objectives.
2. **Intermediate checkpoints.** Between steps you can inspect, validate, or transform the intermediate result before it flows downstream.
3. **Focused generation parameters.** Different steps may benefit from different temperatures — creative generation steps can run hotter, while classification or extraction steps should be near-deterministic.

The key engineering question is: *Is the quality gain worth the added latency and token cost?* We will explore this question empirically in the "Chains vs. Single Long Prompt" section later.

## Setup

Let's start by installing dependencies, loading the quantized model, and defining our helper function for generation.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import re
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Prompt Chaining Utility

A lightweight helper that threads results through a sequence of prompt steps — no framework needed.

In [ ]:
def chain_prompts(*steps):
    """Execute a sequence of prompt steps, passing each output to the next."""
    result = None
    for i, (system, user_template) in enumerate(steps):
        if result:
            user_msg = user_template.format(previous=result)
        else:
            user_msg = user_template
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user_msg}]
        result = generate(messages)
        print(f"\n--- Step {i+1} ---\n{result[:200]}...")
    return result

## Basic Prompt Chaining

Let's start with a simple example of prompt chaining. We'll create two prompts: one to generate a short story, and another to summarize it.

In [ ]:
def story_chain(genre):
    """Generate a story and its summary based on a given genre.

    Args:
        genre (str): The genre of the story to generate.

    Returns:
        tuple: A tuple containing the generated story and its summary.
    """
    story_messages = [
        {"role": "system", "content": "You are a creative writer."},
        {"role": "user", "content": f"Write a short {genre} story in 3-4 sentences."},
    ]
    story = generate(story_messages)

    summary_messages = [
        {"role": "system", "content": "You are a concise editor."},
        {"role": "user", "content": f"Summarize the following story in one sentence:\n{story}"},
    ]
    summary = generate(summary_messages)

    return story, summary

# Test the chain
genre = "science fiction"
story, summary = story_chain(genre)
print(f"Story: {story}\n\nSummary: {summary}")

## Context Management in Chains

Every LLM has a finite **context window** — the maximum number of tokens it can process in a single forward pass. When you chain prompts, each step's input is composed of three parts:

1. **Instruction tokens** — the system prompt and user-facing task description for this step.
2. **Context tokens** — output carried over from previous steps (the "memory" of the chain).
3. **Generation budget** — the remaining tokens available for the model's response.

```
|<---------- context window (e.g. 32,768 tokens) ---------->|
| instruction | context from prev steps |  generation budget |
```

The problem: if previous steps produce verbose output, the context-from-previous-steps portion grows with every hop. After enough hops, there is no room left for generation — or the model is forced to attend over a massive context filled with redundant detail, degrading quality.

This is the **context budget problem**, and it is the central systems-engineering challenge in prompt chaining.

In [ ]:
# Demonstrate context growth in a chain with verbose intermediate outputs

def verbose_chain_demo(topic, num_steps=4):

    """Run a chain where each step elaborates on the previous output.
    Track token counts at each step to show context growth."""

    results = []
    token_counts = []
    current_input = topic

    for i in range(num_steps):
        system = "You are a thorough researcher. Provide detailed, comprehensive analysis."
        user = (
            f"Expand on the following with detailed analysis, examples, and "
            f"supporting arguments (aim for a thorough response):\n\n{current_input}"
        )
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]

        # Count tokens in the input to the model
        input_text = tokenizer.apply_chat_template(messages, tokenize=False,
                                                    add_generation_prompt=True,
                                                    enable_thinking=False)
        input_tokens = len(tokenizer.encode(input_text))

        result = generate(messages, max_new_tokens=300)
        output_tokens = len(tokenizer.encode(result))

        token_counts.append({
            "step": i + 1,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cumulative_context": input_tokens + output_tokens,
        })
        results.append(result)
        current_input = result  # Pass full verbose output to next step

        print(f"\n--- Step {i+1} ---")
        print(f"  Input tokens:  {input_tokens:,}")
        print(f"  Output tokens: {output_tokens:,}")
        print(f"  Output preview: {result[:120]}...")

    # Show the growth trend
    print("\n" + "=" * 50)
    print("CONTEXT GROWTH SUMMARY")
    print("=" * 50)
    for tc in token_counts:
        bar = "█" * (tc["input_tokens"] // 50)
        print(f"  Step {tc['step']}: {tc['input_tokens']:>5} input tokens {bar}")

    growth = token_counts[-1]["input_tokens"] / max(token_counts[0]["input_tokens"], 1)
    print(f"\n  Context grew {growth:.1f}x from step 1 to step {num_steps}.")
    if growth > 3:
        print("  ⚠ This growth rate would overflow the context window in a longer chain!")
    return results, token_counts

verbose_results, verbose_counts = verbose_chain_demo("The impact of renewable energy on global economics")

In [ ]:
# Solution: Summarize intermediate outputs before passing them forward

def compressed_chain_demo(topic, num_steps=4):
    """Same chain, but compress each intermediate output before passing it on."""

    results = []
    token_counts = []
    current_input = topic

    for i in range(num_steps):
        # Step A: Generate detailed content
        system = "You are a thorough researcher. Provide detailed, comprehensive analysis."
        user = (
            f"Expand on the following with detailed analysis, examples, and "
            f"supporting arguments (aim for a thorough response):\n\n{current_input}"
        )
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        result = generate(messages, max_new_tokens=300)

        # Step B: Compress the output into key points only
        compress_messages = [
            {"role": "system", "content": "You are a concise summarizer. Extract only the essential points."},
            {"role": "user", "content": (
                f"Distill the following into 2-3 bullet points capturing ONLY the key "
                f"claims and evidence. Remove all filler and repetition:\n\n{result}"
            )},
        ]
        compressed = generate(compress_messages, max_new_tokens=150)

        # Measure token counts on the compressed version going forward
        next_input_text = tokenizer.apply_chat_template(
            [{"role": "system", "content": system},
             {"role": "user", "content": f"Expand on:\n\n{compressed}"}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        input_tokens = len(tokenizer.encode(next_input_text))

        token_counts.append({"step": i + 1, "input_tokens": input_tokens})
        results.append({"full": result, "compressed": compressed})
        current_input = compressed  # Pass COMPRESSED output forward

        print(f"\n--- Step {i+1} ---")
        print(f"  Full output tokens:   {len(tokenizer.encode(result)):,}")
        print(f"  Compressed to tokens: {len(tokenizer.encode(compressed)):,}")
        print(f"  Next-step input:      {input_tokens:,} tokens")

    # Compare with the verbose version
    print("\n" + "=" * 50)
    print("COMPRESSION COMPARISON")
    print("=" * 50)
    for i, tc in enumerate(token_counts):
        verbose_input = verbose_counts[i]["input_tokens"] if i < len(verbose_counts) else 0
        savings = ((verbose_input - tc["input_tokens"]) / max(verbose_input, 1)) * 100
        print(f"  Step {tc['step']}: {tc['input_tokens']:>5} tokens (vs {verbose_input:>5} verbose, {savings:.0f}% savings)")

compressed_results, compressed_counts = compressed_chain_demo("The impact of renewable energy on global economics")

### The Fundamental Tension

Context management in chains presents an inherent tension:

- **Too much detail** → context overflow, degraded attention, wasted tokens.
- **Too much compression** → information loss, downstream steps lack the nuance they need.

There is no universal right answer. The appropriate level of compression depends on:

1. **How many remaining steps** need the information (more steps → compress more aggressively).
2. **What type of information** is critical (factual claims survive compression better than stylistic nuance).
3. **The model's context window size** (larger windows give you more budget, but don't eliminate the problem — they just push the cliff further out).

A practical heuristic: keep intermediate outputs to **≤25% of the context window**, leaving room for the instruction and a generous generation budget. Compress whenever an intermediate result exceeds this threshold.

## Graceful Failure in Chains

A chain is a pipeline, and like any pipeline, it is only as reliable as its weakest stage. A single step that returns malformed output, hallucinates an unexpected format, or silently drops key information can corrupt every downstream result.

Robust chains need three mechanisms:

1. **Output validation** — after each step, check that the output meets structural and semantic expectations.
2. **Retry with variation** — if validation fails, re-run the step with a modified prompt (e.g., more explicit format instructions).
3. **Fallback strategies** — if retries are exhausted, degrade gracefully (use a default, skip the step, or alert the user) rather than propagating garbage downstream.

In [ ]:
# Robust chain with validation, retry, and fallback at every step

import json

def validated_step(system, user, validator_fn, step_name,
                   max_retries=3, max_new_tokens=512):
    """Execute a single chain step with validation and retries.

    Args:
        system: System prompt for the step.
        user: User prompt for the step.
        validator_fn: Callable that takes the output string and returns
                      (is_valid: bool, parsed_result_or_error_msg: Any).
        step_name: Human-readable name for logging.
        max_retries: How many times to retry on validation failure.
        max_new_tokens: Token budget for generation.

    Returns:
        The validated result, or raises RuntimeError after exhausting retries.
    """
    for attempt in range(1, max_retries + 1):
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": user}]
        raw = generate(messages, max_new_tokens=max_new_tokens)

        is_valid, result = validator_fn(raw)
        if is_valid:
            print(f"  ✓ [{step_name}] passed validation on attempt {attempt}")
            return result

        print(f"  ✗ [{step_name}] attempt {attempt} failed: {result}")
        # Tighten the prompt on retry by appending the error feedback
        user = (
            f"{user}\n\n[IMPORTANT: Your previous response was invalid. "
            f"Error: {result}. Please follow the format instructions exactly.]"
        )

    raise RuntimeError(f"[{step_name}] failed after {max_retries} attempts")


def validate_json_with_keys(required_keys):
    """Return a validator that checks for valid JSON with required keys."""
    def validator(text):
        # Try to extract JSON from the response (model might wrap it in markdown)
        json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if not json_match:
            return False, "No JSON object found in response"
        try:
            data = json.loads(json_match.group())
        except json.JSONDecodeError as e:
            return False, f"Invalid JSON: {e}"
        missing = [k for k in required_keys if k not in data]
        if missing:
            return False, f"Missing keys: {missing}"
        return True, data
    return validator


# Demo: a two-step chain where each step must produce valid JSON
print("=" * 50)
print("VALIDATED CHAIN DEMO")
print("=" * 50)

# Step 1: Extract entities as JSON
text = "Apple CEO Tim Cook announced a new partnership with NASA to develop AR tools for astronaut training."

step1_result = validated_step(
    system="You extract named entities from text. Respond with ONLY a JSON object.",
    user=(
        f"Extract all named entities from this text and categorize them. "
        f"Respond with a JSON object having keys: \"people\", \"organizations\", \"topics\". "
        f"Each value should be a list of strings.\n\nText: {text}"
    ),
    validator_fn=validate_json_with_keys(["people", "organizations", "topics"]),
    step_name="Entity Extraction",
    max_new_tokens=200,
)
print(f"  Entities: {step1_result}")

# Step 2: Generate a summary using the extracted entities
step2_result = validated_step(
    system="You are a news summarizer. Respond with ONLY a JSON object.",
    user=(
        f"Given these entities: {json.dumps(step1_result)}, write a one-sentence summary "
        f"of the news. Respond with a JSON object having keys: \"summary\", \"category\". "
        f"Category should be one of: tech, science, politics, business."
    ),
    validator_fn=validate_json_with_keys(["summary", "category"]),
    step_name="Summary Generation",
    max_new_tokens=200,
)
print(f"  Summary: {step2_result}")

In [ ]:
# Demonstrate failure detection and recovery in a chain

def chain_with_fallback(text):
    """A chain that gracefully handles failures at each step.

    Step 1: Classify the text into a category (must be one of a fixed set).
    Step 2: Based on category, apply a category-specific analysis.
    Fallback: If classification fails, use a generic analysis."""

    VALID_CATEGORIES = {"technical", "narrative", "persuasive", "informational"}

    # --- Step 1: Classification with validation ---
    category = None
    for attempt in range(3):
        messages = [
            {"role": "system", "content": "You are a text classifier. Respond with ONLY one word."},
            {"role": "user", "content": (
                f"Classify this text into exactly ONE category: {VALID_CATEGORIES}.\n"
                f"Respond with ONLY the category word, nothing else.\n\nText: {text}"
            )},
        ]
        raw = generate(messages, max_new_tokens=10, temperature=0.3)
        cleaned = raw.strip().lower().rstrip(".")

        if cleaned in VALID_CATEGORIES:
            category = cleaned
            print(f"  ✓ Classification succeeded: '{category}' (attempt {attempt + 1})")
            break
        print(f"  ✗ Attempt {attempt + 1}: got '{cleaned}', expected one of {VALID_CATEGORIES}")

    # --- Fallback if classification failed ---
    if category is None:
        print("  ⚠ Classification failed — falling back to generic analysis")
        category = "informational"  # safe default

    # --- Step 2: Category-specific analysis ---
    analysis_prompts = {
        "technical": "Evaluate the technical accuracy and identify any claims that need citations.",
        "narrative": "Analyze the narrative structure: protagonist, conflict, and resolution.",
        "persuasive": "Identify the rhetorical strategies used: ethos, pathos, logos.",
        "informational": "Summarize the key facts and assess the completeness of coverage.",
    }

    messages = [
        {"role": "system", "content": f"You are a {category}-text analyst."},
        {"role": "user", "content": f"{analysis_prompts[category]}\n\nText: {text}"},
    ]
    analysis = generate(messages, max_new_tokens=300)
    print(f"  Category: {category}")
    print(f"  Analysis: {analysis[:200]}...")
    return {"category": category, "analysis": analysis}


# Test with a technical text
print("=" * 50)
print("FAILURE RECOVERY DEMO")
print("=" * 50)
tech_text = (
    "The transformer architecture uses multi-head self-attention mechanisms "
    "with O(n²) complexity relative to sequence length. Recent advances in "
    "linear attention reduce this to O(n) but with some quality tradeoffs."
)
result = chain_with_fallback(tech_text)

# Test with an ambiguous text that may challenge the classifier
print("\n" + "=" * 50)
print("AMBIGUOUS TEXT TEST")
print("=" * 50)
ambiguous = "Once upon a time, researchers discovered that storytelling improves data retention by 22x."
result2 = chain_with_fallback(ambiguous)

### Reliability Principles for Production Chains

In production chains, every step should validate both its **input** (is the data from the previous step well-formed?) and its **output** (does my result match the expected schema?). Key principles:

- **Fail fast, fail loud.** If a step detects invalid input, raise an error immediately rather than attempting to work with corrupted data. Silent failures propagate.
- **Retry with escalation.** On the first retry, add explicit format instructions. On the second, simplify the task. On the third, use a fallback.
- **Log everything.** In a multi-step chain, debugging requires knowing exactly what each step received and produced. Structured logging (step name, attempt number, input/output tokens, validation result) is essential.
- **The chain is only as reliable as its weakest link.** Invest validation effort proportionally to how much damage a step's failure causes downstream.

## Sequential Prompting

Now, let's create a more complex sequence of prompts for a multi-step analysis task. We'll analyze a given text for its main theme, tone, and key takeaways.

In [ ]:
def analyze_text(text):
    """Perform a multi-step analysis of a given text.

    Args:
        text (str): The text to analyze.

    Returns:
        dict: A dictionary containing the theme, tone, and key takeaways of the text.
    """
    # Step 1: Identify theme
    theme_messages = [
        {"role": "system", "content": "You are a literary analyst."},
        {"role": "user", "content": f"Identify the main theme of the following text:\n{text}"},
    ]
    theme = generate(theme_messages)

    # Step 2: Identify tone
    tone_messages = [
        {"role": "system", "content": "You are a literary analyst."},
        {"role": "user", "content": f"Describe the overall tone of the following text:\n{text}"},
    ]
    tone = generate(tone_messages)

    # Step 3: Extract takeaways using previous results
    takeaway_messages = [
        {"role": "system", "content": "You are a literary analyst."},
        {"role": "user", "content": (
            f"Given the following text with the main theme '{theme}' "
            f"and tone '{tone}', what are the key takeaways?\n{text}"
        )},
    ]
    takeaways = generate(takeaway_messages)

    return {"theme": theme, "tone": tone, "takeaways": takeaways}

# Test the sequential prompting
sample_text = (
    "The rapid advancement of artificial intelligence has sparked both excitement "
    "and concern among experts. While AI promises to revolutionize industries and "
    "improve our daily lives, it also raises ethical questions about privacy, job "
    "displacement, and the potential for misuse. As we stand on the brink of this "
    "technological revolution, it is crucial that we approach AI development with "
    "caution and foresight, ensuring that its benefits are maximized while its "
    "risks are minimized."
)

analysis = analyze_text(sample_text)
for key, value in analysis.items():
    print(f"{key.capitalize()}: {value}\n")

## Dynamic Prompt Generation

In this section, we'll create a dynamic question-answering system that generates follow-up questions based on previous answers.

In [ ]:
def dynamic_qa(initial_question, num_follow_ups=3):
    """Conduct a dynamic Q&A session with follow-up questions.

    Args:
        initial_question (str): The initial question to start the Q&A session.
        num_follow_ups (int): The number of follow-up questions to generate.

    Returns:
        list: A list of dictionaries containing questions and answers.
    """
    qa_chain = []
    current_question = initial_question

    for i in range(num_follow_ups + 1):  # +1 for the initial question
        answer_messages = [
            {"role": "system", "content": "You are a knowledgeable assistant."},
            {"role": "user", "content": f"Answer the following question concisely:\n{current_question}"},
        ]
        answer = generate(answer_messages)
        qa_chain.append({"question": current_question, "answer": answer})

        if i < num_follow_ups:  # Generate follow-up for all but the last iteration
            follow_up_messages = [
                {"role": "system", "content": "You are a curious researcher."},
                {"role": "user", "content": (
                    f"Based on the question '{current_question}' and the answer "
                    f"'{answer}', generate a relevant follow-up question."
                )},
            ]
            current_question = generate(follow_up_messages)

    return qa_chain

# Test the dynamic Q&A system
initial_question = "What are the potential applications of quantum computing?"
qa_session = dynamic_qa(initial_question)

for i, qa in enumerate(qa_session):
    print(f"Q{i+1}: {qa['question']}")
    print(f"A{i+1}: {qa['answer']}\n")

## Chain Patterns: A Taxonomy

Just as software engineering has design patterns, prompt chaining has recurring structural patterns. Each pattern suits different problem shapes. Understanding when to reach for which pattern is a core prompt-engineering skill.

### Pattern 1: Conditional Branching

The output of an early step determines which downstream path the chain follows. This is essential when different categories of input require fundamentally different processing logic — not just different prompts, but different *chains*.

In [ ]:
# Pattern: Conditional Branching Chain
# Step 1 classifies the input, Step 2 branches to a specialized handler

def conditional_branch_chain(user_query):
    """Route a user query to the appropriate specialized handler based on intent."""

    # Step 1: Classify intent
    classify_messages = [
        {"role": "system", "content": "You classify user queries. Respond with ONLY one word."},
        {"role": "user", "content": (
            f"Classify this query into one of: factual, creative, analytical.\n"
            f"Respond with ONLY one word.\n\nQuery: {user_query}"
        )},
    ]
    intent = generate(classify_messages, max_new_tokens=10, temperature=0.2).strip().lower().rstrip(".")
    print(f"Detected intent: '{intent}'")

    # Step 2: Branch to specialized handler
    branch_configs = {
        "factual": {
            "system": "You are an encyclopedia. Provide accurate, well-sourced answers.",
            "style": "Answer factually with specific data points and dates.",
        },
        "creative": {
            "system": "You are an imaginative storyteller.",
            "style": "Respond creatively with vivid language and original ideas.",
        },
        "analytical": {
            "system": "You are a critical analyst. Examine all sides of an issue.",
            "style": "Provide a structured analysis with pros, cons, and a reasoned conclusion.",
        },
    }

    config = branch_configs.get(intent, branch_configs["analytical"])  # fallback
    messages = [
        {"role": "system", "content": config["system"]},
        {"role": "user", "content": f"{config['style']}\n\nQuery: {user_query}"},
    ]
    response = generate(messages, max_new_tokens=300)
    print(f"\nResponse ({intent} branch):\n{response[:300]}...")
    return {"intent": intent, "response": response}


# Test with different query types
print("--- Factual query ---")
conditional_branch_chain("What is the speed of light in a vacuum?")

print("\n--- Creative query ---")
conditional_branch_chain("Write me a haiku about a robot learning to love.")

print("\n--- Analytical query ---")
conditional_branch_chain("Should companies adopt a four-day work week?")

### Pattern 2: Iterative Refinement

A **generate → critique → improve** loop. The model produces an initial draft, then a separate step (or the same model with a different persona) identifies weaknesses, and a third step addresses those weaknesses. This cycle can repeat until quality converges or a budget is exhausted.

This pattern works because LLMs are often better at *evaluating* text than *generating* perfect text on the first try. The critique step externalizes the model's latent quality signal.

In [ ]:
# Pattern: Iterative Refinement — Generate, Critique, Improve

def iterative_refinement(task, num_iterations=2):
    """Refine output through generate-critique-improve cycles.

    Args:
        task: The writing task to accomplish.
        num_iterations: Number of critique-improve cycles.
    """
    # Initial generation
    messages = [
        {"role": "system", "content": "You are a skilled writer."},
        {"role": "user", "content": task},
    ]
    draft = generate(messages, max_new_tokens=400)
    print("=== Initial Draft ===")
    print(draft[:300], "...\n")

    for i in range(num_iterations):
        print(f"=== Iteration {i+1}: Critique ===")

        # Critique step — identify specific weaknesses
        critique_messages = [
            {"role": "system", "content": (
                "You are a demanding writing critic. Identify specific, actionable "
                "weaknesses. Be concrete — quote the problematic passages."
            )},
            {"role": "user", "content": (
                f"Critique this draft. List exactly 3 specific weaknesses with "
                f"concrete suggestions for improvement:\n\n{draft}"
            )},
        ]
        critique = generate(critique_messages, max_new_tokens=300)
        print(critique[:300], "...\n")

        # Improve step — address the critique
        print(f"=== Iteration {i+1}: Revision ===")
        improve_messages = [
            {"role": "system", "content": "You are a skilled writer revising your work."},
            {"role": "user", "content": (
                f"Revise this draft to address ALL of the following critique. "
                f"Preserve what works, fix what doesn't.\n\n"
                f"DRAFT:\n{draft}\n\n"
                f"CRITIQUE:\n{critique}\n\n"
                f"Write the improved version:"
            )},
        ]
        draft = generate(improve_messages, max_new_tokens=400)
        print(draft[:300], "...\n")

    print("=== Final Version ===")
    print(draft)
    return draft


result = iterative_refinement(
    "Write a concise paragraph explaining why the sky is blue, "
    "suitable for a curious 10-year-old. It should be scientifically "
    "accurate but use simple, vivid language."
)

### Pattern 3: Map-Reduce

When the input is too large or too complex to process in a single call, **split** it into independent chunks, **map** a processing step over each chunk in isolation, then **reduce** (merge) the results into a coherent whole.

This pattern directly leverages the DAG insight from earlier: the map steps have no dependencies on each other and are fully parallelizable. The reduce step depends on all map steps completing.

```
  [chunk1] ──> [analyze] ──┐
  [chunk2] ──> [analyze] ──┼──> [merge/synthesize]
  [chunk3] ──> [analyze] ──┘
```

In [ ]:
# Pattern: Map-Reduce Chain
# Split a complex topic into sub-topics, analyze each, then synthesize

def map_reduce_chain(topic, num_aspects=3):
    """Analyze a complex topic by decomposing it, analyzing parts, then synthesizing."""

    # Step 1: Decompose — Split the topic into independent sub-questions
    decompose_messages = [
        {"role": "system", "content": "You decompose complex topics into sub-questions."},
        {"role": "user", "content": (
            f"Break the following topic into exactly {num_aspects} independent sub-questions "
            f"that together cover the full scope. List them as numbered items, one per line.\n\n"
            f"Topic: {topic}"
        )},
    ]
    decomposition = generate(decompose_messages, max_new_tokens=200)
    print("=== Decomposition ===")
    print(decomposition, "\n")

    # Parse the sub-questions (simple line-based parsing)
    sub_questions = [
        line.strip().lstrip("0123456789.-) ") for line in decomposition.strip().split("\n")
        if line.strip() and any(c.isalpha() for c in line)
    ][:num_aspects]

    # Step 2: Map — Analyze each sub-question independently
    analyses = []
    for i, sq in enumerate(sub_questions):
        print(f"=== Map Step {i+1}: {sq[:60]}... ===")
        map_messages = [
            {"role": "system", "content": "You are a subject-matter expert. Be specific and concise."},
            {"role": "user", "content": f"Provide a focused analysis of this question:\n{sq}"},
        ]
        analysis = generate(map_messages, max_new_tokens=250)
        analyses.append({"question": sq, "analysis": analysis})
        print(f"  {analysis[:150]}...\n")

    # Step 3: Reduce — Synthesize all analyses into a cohesive answer
    combined = "\n\n".join(
        f"Sub-question: {a['question']}\nAnalysis: {a['analysis']}"
        for a in analyses
    )
    reduce_messages = [
        {"role": "system", "content": "You synthesize multiple analyses into a coherent whole."},
        {"role": "user", "content": (
            f"Synthesize these {len(analyses)} independent analyses into a single, "
            f"cohesive response about: {topic}\n"
            f"Identify common themes, resolve any contradictions, and provide "
            f"an integrated conclusion.\n\n{combined}"
        )},
    ]
    synthesis = generate(reduce_messages, max_new_tokens=400)
    print("=== Reduce: Synthesis ===")
    print(synthesis)
    return {"sub_analyses": analyses, "synthesis": synthesis}


result = map_reduce_chain("The long-term societal implications of remote work becoming the default")

## Chains vs. a Single Long Prompt

Is chaining always better? No. Chaining introduces **information loss at boundaries** (each step only sees what you explicitly pass forward), **added latency** (N sequential model calls), and **compounding error risk** (each step can fail). A single well-crafted prompt avoids all of these.

Let's empirically compare the two approaches on the same complex task.

In [ ]:
# Compare: Single long prompt vs. a 3-step chain for the same complex task

TASK_TEXT = (
    "Artificial intelligence is transforming healthcare through improved diagnostics, "
    "personalized treatment plans, and drug discovery. However, challenges remain in "
    "data privacy, algorithmic bias, and the need for regulatory frameworks. Recent "
    "breakthroughs in foundation models have accelerated progress, but concerns about "
    "hallucination and reliability in clinical settings persist."
)

COMPLEX_TASK = (
    f"Analyze this text by: (1) identifying the 3 most important claims, "
    f"(2) evaluating the strength of evidence for each claim, and "
    f"(3) suggesting what additional evidence would be needed to fully support each claim.\n\n"
    f"Text: {TASK_TEXT}"
)

# --- Approach A: Single monolithic prompt ---
print("=" * 60)
print("APPROACH A: Single Long Prompt")
print("=" * 60)

mono_messages = [
    {"role": "system", "content": "You are a critical analyst."},
    {"role": "user", "content": COMPLEX_TASK},
]
mono_result = generate(mono_messages, max_new_tokens=600)
mono_tokens = len(tokenizer.encode(mono_result))
print(f"Output ({mono_tokens} tokens):\n{mono_result}\n")

# --- Approach B: 3-step chain ---
print("=" * 60)
print("APPROACH B: 3-Step Chain")
print("=" * 60)

# Step 1: Extract claims
print("\n--- Step 1: Extract Claims ---")
step1_msgs = [
    {"role": "system", "content": "You extract claims from text. Be precise and exhaustive."},
    {"role": "user", "content": (
        f"Identify the 3 most important factual claims in this text. "
        f"List each as a single, precise sentence.\n\nText: {TASK_TEXT}"
    )},
]
claims = generate(step1_msgs, max_new_tokens=200)
print(claims)

# Step 2: Evaluate evidence
print("\n--- Step 2: Evaluate Evidence ---")
step2_msgs = [
    {"role": "system", "content": "You evaluate the strength of evidence for claims."},
    {"role": "user", "content": (
        f"For each of these claims, rate the strength of evidence as Strong, "
        f"Moderate, or Weak, and explain your reasoning:\n\n{claims}"
    )},
]
evidence = generate(step2_msgs, max_new_tokens=300)
print(evidence)

# Step 3: Suggest additional evidence needed
print("\n--- Step 3: Evidence Gaps ---")
step3_msgs = [
    {"role": "system", "content": "You identify what evidence is missing to support claims."},
    {"role": "user", "content": (
        f"For each claim and its evidence evaluation below, suggest specific "
        f"additional evidence that would strengthen the case.\n\n"
        f"Claims:\n{claims}\n\nEvidence evaluation:\n{evidence}"
    )},
]
gaps = generate(step3_msgs, max_new_tokens=300)
chain_tokens = sum(len(tokenizer.encode(x)) for x in [claims, evidence, gaps])
print(gaps)

# --- Comparison ---
print("\n" + "=" * 60)
print("COMPARISON")
print("=" * 60)
print(f"  Single prompt:  {mono_tokens:>4} output tokens, 1 model call")
print(f"  3-step chain:   {chain_tokens:>4} output tokens, 3 model calls")
print()
print("  Examine both outputs above. The chain typically produces more")
print("  structured, detailed analysis because each step can focus fully")
print("  on one sub-task. The single prompt may miss nuances or produce")
print("  a more superficial treatment of each aspect.")

### When to Chain vs. When to Use a Single Prompt

**Chain when:**
1. The task has **distinct phases** with different cognitive requirements (e.g., extract → evaluate → synthesize).
2. You need **intermediate validation** — to check or transform results between steps.
3. Different steps benefit from **different generation parameters** (temperature, max tokens, system persona).
4. The task is **too complex** for a single prompt to handle reliably — the model loses coherence or drops sub-tasks.

**Use a single prompt when:**
1. The task is **simple enough** that a single prompt handles it well.
2. Steps are **tightly coupled** — the output of each "step" depends intimately on the others, making it hard to decompose without losing coherence.
3. **Information loss at boundaries** would hurt quality — subtle stylistic or contextual cues that are hard to carry across prompt boundaries.
4. **Latency is critical** — one call is always faster than N calls.

As a rule of thumb: start with a single prompt. If the output quality is insufficient and you can identify distinct sub-tasks, refactor into a chain. Don't chain prematurely — it adds complexity, cost, and latency.

## Error Handling and Validation

In this final section, we implement error handling and validation in our prompt chains to make them more robust.

In [ ]:
def extract_number(text):
    """Extract a 4-digit number from the given text.

    Args:
        text (str): The text to extract the number from.

    Returns:
        str or None: The extracted 4-digit number, or None if no valid number is found.
    """
    match = re.search(r'\b\d{4}\b', text)
    return match.group() if match else None

def robust_number_generation(topic, max_attempts=3):
    """Generate a topic-related number with validation and error handling.

    Args:
        topic (str): The topic to generate a number for.
        max_attempts (int): Maximum number of generation attempts.

    Returns:
        str: A validated 4-digit number or an error message.
    """
    for attempt in range(max_attempts):
        try:
            gen_messages = [
                {"role": "system", "content": "You respond with only a number."},
                {"role": "user", "content": (
                    f"Generate a 4-digit number related to the topic: {topic}. "
                    "Respond with ONLY the number, no additional text."
                )},
            ]
            response = generate(gen_messages, max_new_tokens=16)
            number = extract_number(response)

            if not number:
                raise ValueError(f"Failed to extract a 4-digit number from the response: {response}")

            # Validate the relevance
            val_messages = [
                {"role": "system", "content": "You are a fact checker."},
                {"role": "user", "content": (
                    f"Is the number {number} truly related to the topic '{topic}'? "
                    "Answer with 'Yes' or 'No' and explain why."
                )},
            ]
            validation = generate(val_messages)
            if validation.lower().startswith("yes"):
                return number
            else:
                print(f"Attempt {attempt + 1}: Number {number} was not validated. Reason: {validation}")
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {str(e)}")

    return "Failed to generate a valid number after multiple attempts."

# Test the robust number generation
topic = "World War II"
result = robust_number_generation(topic)
print(f"Final result for topic '{topic}': {result}")

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** rewrite a prompt using the technique from this notebook. Document what changes and why.

**Exercise 2 — Build:** test the technique on a different task and compare results. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** combine this technique with one from a previous notebook.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the prompt-engineering/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [Negative-Prompting](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/negative-prompting.ipynb) | ➡️ **Next:** [Prompt-Formatting-Structure](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/prompt-formatting-structure.ipynb)